# Encoding Categorical Features

## Learning objectives
- Understand why many estimators need numeric inputs.
- Use one-hot encoding for nominal categories.
- Use ordinal encoding only when categories have a meaningful order.

In [1]:
import pandas as pd

df = pd.DataFrame({
    "City": ["Amsterdam", "Berlin", "Amsterdam", "Paris", "Berlin"],
    "Education": ["Bachelor", "Master", "PhD", "Bachelor", "Master"],
    "Experience": [2, 5, 8, 1, 6],
})
df

,City,Education,Experience
0,Amsterdam,Bachelor,2
1,Berlin,Master,5
2,Amsterdam,PhD,8
3,Paris,Bachelor,1
4,Berlin,Master,6


## 1. One-hot encoding
One-hot encoding creates a binary column for each category. It is appropriate for nominal categories with no natural order.

In [2]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(df[["City"]])
encoded_df = pd.DataFrame(
    encoded, columns=encoder.get_feature_names_out(["City"]), index=df.index
)
encoded_df

,City_Amsterdam,City_Berlin,City_Paris
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,0.0,1.0,0.0


`handle_unknown="ignore"` lets the encoder handle categories not seen during fitting.

In [3]:
encoder.transform(pd.DataFrame({"City": ["Berlin", "Rome"]}))

array([[0., 1., 0.],
       [0., 0., 0.]])

## 2. Ordinal encoding
Use ordinal encoding only when there is a meaningful order, and specify that order explicitly.

In [4]:
from sklearn.preprocessing import OrdinalEncoder

order = [["HighSchool", "Bachelor", "Master", "PhD"]]
ordinal_encoder = OrdinalEncoder(categories=order)
education = pd.DataFrame({"Education": ["Bachelor", "PhD", "HighSchool", "Master"]})
education["Education_Level"] = ordinal_encoder.fit_transform(education[["Education"]])
education

,Education,Education_Level
0,Bachelor,1.0
1,PhD,3.0
2,HighSchool,0.0
3,Master,2.0


## 3. Encoding in a model pipeline

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X = df[["City", "Experience"]]
y = pd.Series([0, 1, 1, 0, 1], name="Purchased")
preprocessor = ColumnTransformer([
    ("city", OneHotEncoder(handle_unknown="ignore"), ["City"]),
    ("experience", "passthrough", ["Experience"]),
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
model.fit(X_train, y_train)
print("Predictions:", model.predict(X_test))
print("Accuracy:", model.score(X_test, y_test))

Predictions: [1 0]
Accuracy: 1.0


## Key takeaways
- One-hot encode nominal categories.
- Ordinal encode only meaningful ordered categories.
- Keep transformations in a pipeline so they are fitted on training data.